# 1. Importacion de Librerias 
---
1. Importa librerías para trabajar con web scraping, bases de datos y manejo de archivos.
2. Configura una sesión HTTP con reintentos automáticos en caso de errores de red.
3. Crea un directorio datos para guardar la información obtenida.
4. Muestra mensajes confirmando que todo se configuró correctamente.
***
- preparar el entorno, y asegura el lugar,
- importar herramientas necesarias el codigo antes de scrapear.

In [1]:
# 1.1 Importacion de librerias necesarias
import requests
from bs4 import BeautifulSoup
import sqlite3
import time
import json
import re
from urllib.parse import urljoin, urlparse, quote
from datetime import datetime
import pandas as pd
import os
from typing import Dict,List,Optional,Tuple,Any
import hashlib

#1.2 para manejar errores en red 
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

#1.3 Configuracion de sesion con reintentos
sesion = requests.Session()
sesion.headers.update({
    'User-Agent': 'web-scraping-challenge/1.0 (educational project)'
})
retry=Retry (total=3,
            backoff_factor=0.5,
            status_forcelist=[429,500,502,503,504])

#1.4 adaptador
adaptador=HTTPAdapter(max_retries=retry)
sesion.mount('http://', adaptador)
sesion.mount('https://', adaptador)

#1.5 crear carpetas datos si no existe
try:
    os.makedirs('datos', exist_ok=True)
    print("Directorio 'datos' Verificado/creado Correctamente.")
except PermissionError:
    print("Error: No tienes permisos para crear el directorio 'datos'.")
except FileNotFoundError:
    print("Error: la ruta especificada no es válida o alguna carpeta padre no existe")
except OSError as e:
    print(f"Error al crear el directorio {e}")

#1.6 aviso de librerias importadas correctamente 
print("Configuracion completada. librerias importadas")

Directorio 'datos' Verificado/creado Correctamente.
Configuracion completada. librerias importadas


# 2. Funciones auxiliares (HTML, espera, cache)
1. __`obtener_html(url, timeout=10):`__
Obtiene el contenido de una página web con requests, convierte en un objeto BeautifulSoup para analizar y, si ocurre un error de red, devuelve None.
2. __`esperar(segundos=1):`__
pausa durante el tiempo indicado y evitar realizar demasiadas peticiones seguidas al servidor.
3. __`obtener_conexion():`__
Abre la base de datos SQLite libreria.db, activa las claves foráneas y devuelve la conexión.
4. __`cache_autores y MARCADOR_NO_ENCONTRADO:`__
- `cache_autores:` guarda en memoria datos de autores ya consultados.
- `MARCADOR_NO_ENCONTRADO:` identifica aquellos autores que no pudieron encontrarse.
5. __`cargar_cache_autores()`__
- Comprueba si hay datos/cache_autores.csv, carga sus registros en cache_autores y convierte los valores vacíos en None.
6. __`guardar_cache_autores()`__
Guarda el contenido de cache_autores en datos/cache_autores.csv, permite conservar los datos entre ejecuciones.
7. __`cargar_cache_autores()`__ al inicio
- Se ejecuta automáticamente al comenzar el programa para recuperar el caché y evitar repetir consultas a la API.


In [2]:
#2.1 Obtiene el contenido html de una pagina web, una url y lo devuelve, un objeto beutifulsoup.
def obtener_html(url:str, timeout:int=10)->Optional[BeautifulSoup]:
    try:
        respuesta = sesion.get(url, timeout=timeout)
        respuesta.raise_for_status()
        soup=BeautifulSoup(respuesta.content,'html.parser')
        print(f"Contenido HTML obtenido correctamente de {url}")
        return soup
    except requests.exceptions.HTTPError as e:
        if e.response.status_code==404:
            print(f" Pagina web no encontrada (404): {url}")
        else:
            print(f"Error HTTP Al obtener el contenido HTML {e.response.status_code} de: {url}")
    except requests.exceptions.Timeout:
        print(f"Tiempo de espera agotado al obtener el contenido HTML de: {url}")
    except requests.exceptions.RequestException as e:
        print(f" Error de red al obtener el contenido HTML de: {url}. Detalles: {e}")
    return None

#2.2 funcion para esperar un tiempo pausa para respetar limites de velocidad
def esperar(segundos: float =1):
    print(f" Esperando {segundos} segundos...")
    time.sleep(segundos)

def obtener_conexion():
    conexion=sqlite3.connect('libreria.db')
    conexion.execute ("PRAGMA foreign_keys = ON")
    return conexion

#2.4 cache de autores en (memoria)
cache_autores={}
MARCADOR_NO_ENCONTRADO="NO_ENCONTRADO"

def cargar_cache_autores():
    global cache_autores
    archivo_cache='datos/cache_autores.csv'

    if os.path.exists(archivo_cache):
        df = pd.read_csv(archivo_cache)

        for _, row in df.iterrows():
            nombre=row['nombre']

            if row.get('api_source')== MARCADOR_NO_ENCONTRADO:
                cache_autores[nombre]=None
                continue

            datos={
                'pais': row.get('pais'),
                'api_id': row.get('api_id'),
                'api_source': row.get('api_source')
            }
            for clave, valor in datos.items():
                if pd.isna(valor):
                    datos[clave] = None
            cache_autores[nombre]=datos

        print(f" Cache de autores cargada desde {archivo_cache}. Total autores: ({len(cache_autores)} en registros)")

#2.5 funcion para guardar cache de autores en un archivo csv
def guardar_cache_autores():
    registros=[]
    for nombre, datos in cache_autores.items():
        if datos is None:
            registros.append({
                'nombre': nombre,
                'pais': None,
                'api_id':None,
                'api_source':MARCADOR_NO_ENCONTRADO
            })
            continue

        registros.append({
            'nombre': nombre,
            'pais': datos.get('pais'),
            'api_id':datos.get('api_id'),
            'api_source':datos.get('api_source')
        })

    df=pd.DataFrame(registros, columns=['nombre','pais','api_id','api_source'])
    df.to_csv('datos/cache_autores.csv', index=False, encoding="utf-8")
    print(f" Cache de autores guardada en datos/cache_autores.csv. Total autores: ({len(registros)} en registros)")

#cargar cache al iniciar (para no repetir llamadas a la api)
cargar_cache_autores()

 Cache de autores cargada desde datos/cache_autores.csv. Total autores: (641 en registros)


# 3 Creacion de la  base de datos (DDL "Definicion de Datos Lenguaje")
crea las tablas y relaciones necesarias para organizar categorías, libros y autores, y añade índices para que las consultas sean más rápidas.
***
__`Conexión y cursor`__

- Abre la base de datos y prepara un cursor para ejecutar sentencias SQL.

- Activa la integridad referencial (claves foráneas).

__`Tablas principales`__

- categorias: guarda las categorías de libros (ej. “Ficción”, “Historia”).

- libros: almacena cada libro con título, precio, calificación, categoría y URL.

- autores: contiene los autores, con datos extra como país y fuente de API/Wikipedia.

- autor_libro: tabla intermedia para la relación muchos a muchos (un libro puede tener varios autores y un autor puede tener varios libros).

__`Índices`__

- crean índices en clave (categoria_id, nombre_autor, calificacion_libro, pais_autor) para acelerar las búsquedas.
- Cierra
- Se guardan los cambios (commit), se cierra la conexión y se imprime un mensaje confirmando que todo está listo.


In [3]:
def crear_base_datos():
#3.1 abre la base de datos, preparan un cursor para ejecutar SQL y activan la integridad referencial entre tablas.
    conexion=obtener_conexion()
    cursor=conexion.cursor()

#3.2.1 tabla categorias con sqlite3
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS categorias (
        id_categoria INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre_categoria TEXT UNIQUE NOT NULL,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    ''')

#3.2.2 tabla libros
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS libros(
            id_libro INTEGER PRIMARY KEY AUTOINCREMENT,
            titulo_libro TEXT NOT NULL,
            precio_libro REAL NOT NULL,
            calificacion_libro INTEGER,
            categoria_id INTEGER,
            url_libro TEXT UNIQUE,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY (categoria_id) REFERENCES categorias(id_categoria)
        )
    ''')

#3.2.3 tabla autores (enriquesimiento)
    cursor.execute('''
        CREATE  TABLE IF NOT EXISTS autores(
            id_autor INTEGER PRIMARY KEY AUTOINCREMENT,
            nombre_autor TEXT NOT NULL UNIQUE,
            pais_autor TEXT,
            api_external_id TEXT,
            api_source TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )    
    ''')

#3.2.4 tablas intermedias relacion muchos a muchos
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS autor_libro (
            id_libro INTEGER,
            id_autor INTEGER,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            PRIMARY KEY (id_libro, id_autor),
            FOREIGN KEY (id_libro) REFERENCES libros(id_libro),
            FOREIGN KEY (id_autor) REFERENCES autores(id_autor)
        )
    ''')

#3.3 indices para optimizar consultas
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_libros_categoria ON libros(categoria_id)')
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_autores_nombre ON autores(nombre_autor)')
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_libros_calificacion ON libros(calificacion_libro)')
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_autores_pais ON autores(pais_autor)')

#3.4 commit y cerrar conexion
    conexion.commit()
    conexion.close()
    print(" Base de datos y tablas creadas/verificadas.")

crear_base_datos()

 Base de datos y tablas creadas/verificadas.


# 4. Funciones de scraping ( categori­as, – libros y ¸ autores)
navegan por la web de libros, saca las categorías, listan los libros con sus datos y identifica el autor desde la página del libro.
***
__`extraer_categoria(url_principal)`__

- Descarga la página principal.
- Busca el menú lateral de categorías.
- Extrae el nombre y la URL de cada categoría (excepto la genérica “Books”).
- Devuelve un diccionario con todas las categorías encontradas.

__`extraer_libros_categoria`__(url_categoria, categoria_nombre)

- Recorre todas las páginas de una categoría (incluye paginación).
- Para cada libro obtiene: título, URL, precio y calificación en estrellas.
- Convierte la calificación textual (“One”, “Two”, etc.) en número (1–5).
- Devuelve una lista de diccionarios con los datos de los libros.

__`extraer_autor_desde_libro`__(url_libro)

- Abre la página individual de un libro.
- encuentra el nombre del autor usando estrategias:
1. Buscar etiquetas li class="author".
2. Buscar etiquetas p class="authors".
3. Buscar enlaces cuya URL contenga “author”.
4. Revisar el texto de la página buscando “Author: …”.
- Si encuentra un nombre válido, lo devuelve; si no, devuelve None.


In [4]:
#4.1 Funcion de extraccion extrae todas las categorias y sus URLs
def extraer_categoria(url_principal: str) -> Dict[str,str]:
    soup= obtener_html(url_principal)
    if not soup:
        print(f"-> No se pudo obtener el contenido HTML de la pagina principal: {url_principal}")
        return{}

    categorias={}
    barra_lateral=soup.find('div',class_='side_categories')
    if barra_lateral:
        enlaces=barra_lateral.find_all('a')
        for enlace in enlaces:
            nombre=enlace.text.strip()
            if nombre.lower()!='books':
                url_relativa=enlace.get("href")
                url_completa=urljoin(url_principal,url_relativa)
                categorias[nombre]=url_completa

    print(f"Se encontro: {len(categorias)} Categorias")
    return categorias

#4.2 Extrae todos los libros de una categori­a (con paginacion).
def extraer_libros_categoria(url_categoria:str,categoria_nombre:str)->List[Dict]:
    libros=[]
    pagina_actual=url_categoria

    while pagina_actual:
        soup= obtener_html(pagina_actual)
        if not soup:
            print(f"-> No se pudo obtener el contenido HTML de la pagina principal: {pagina_actual}")
            break

        articulos=soup.find_all('article',class_='product_pod')
        for articulo in articulos:
            enlace_titulos= articulo.find('h3').find('a')
            if (not enlace_titulos):
                print(f"-> No se encontro el enlace del ti­tulo en el arti­culo: {articulo}")
                continue

            titulo=enlace_titulos.get('title')
            url_libro=urljoin(pagina_actual,enlace_titulos.get('href'))

            precio_tag=articulo.find('p', class_='price_color')
            precio_texto=precio_tag.text.strip() if precio_tag else '£0.00'
            precio=float(re.sub(r'[^\d.]', '', precio_texto))

            rating_tag=articulo.find('p',class_='star-rating')

            if rating_tag:
                clases= rating_tag.get('class')
                calificacion_texto= clases[1] if len(clases)>1 else 'Zero'
                mapeo={'Zero':0,'One':1,'Two':2,'Three':3,'Four':4,'Five':5,}
                calificacion=mapeo.get(calificacion_texto,0)
            else:
                calificacion=0

            libros.append({
                'titulo':titulo,
                'url':url_libro,
                'precio':precio,
                'calificacion':calificacion,
                'categoria':categoria_nombre
            })

        siguiente=soup.find('li',class_='next')
        if siguiente:
            enlace_siguiente=siguiente.find('a')
            if enlace_siguiente:
                pagina_actual=urljoin(pagina_actual,enlace_siguiente.get('href'))
                esperar(0.5)
            else:
                pagina_actual=None
        else:
            pagina_actual=None
    print(f" Categoria {categoria_nombre}: {len(libros)} Libros.")
    return libros

# 4.3 Extrae el nombre del autor desde la pÃ¡gina del libro.
def extraer_autor_desde_libro(url_libro: str)-> Optional[str]:
    soup= obtener_html(url_libro)
    if not soup:
        return None

    # Estrategia 1: Buscar <li class="author">
    autor_tag=soup.find('li', class_='author')# si o si en ingles para hacer referencia a la pagina web
    if autor_tag:
        enlace_autor=autor_tag.find('a')
        if enlace_autor:
            return enlace_autor.text.strip()
        # si no hay <a>,tomar texto directamente
        texto=autor_tag.get_text(strip=True)
        if texto and texto!='Author:':
            return texto

    #Estrategia 2 buscar <p class="author"
    autor_tag =soup.find('p',class_='authors')
    if autor_tag:
        enlace_autor=autor_tag.find('a')
        if enlace_autor:
            return enlace_autor.text.strip()
        texto=autor_tag.get_text(strip=True)
        if texto and texto!='Authors:':
            return texto

    #Estrategia 3 Buscar cualquier enlace que contenga "author" en la URL
    for enlace in soup.find_all('a', href=True):
        if 'author' in enlace['href'].lower():
            return enlace.text.strip()

    contenido=soup.get_text()
    if 'Author:' in contenido:
        coincidencia=re.search(r'Author:\s*([^\n]+)',contenido)
        if coincidencia:
            nombre_autor= coincidencia.group(1).strip()
            nombre_autor=nombre_autor.split('|')[0].split('•')[0].strip()
            return nombre_autor

    return None

## 5. Enriquecimiento de autores SOLO Wikipedia
este código limpia títulos, valida nombres y usa Wikipedia para obtener datos extra de los autores (como país y fuente), guardándolos en caché para no repetir búsquedas.
***
__`limpiar_titulo_para_busqueda(titulo)`__

- Quita cosas como “(Serie #N)” al final del título del libro.
- Así el título queda más limpio para buscarlo en Wikipedia.

__`es_nombre_valido(nombre)`__

- Revisa si un texto parece un nombre real de persona.
- Descarta si está vacío, demasiado largo, contiene números o símbolos raros.
- Solo acepta letras, espacios y algunos signos como punto o guion.

__`buscar_autor_por_titulo_wikipedia(titulo_libro)`__

- Usa el título del libro para buscar en Wikipedia.
- Analiza el resumen de la página, para encontrar frases como “written by John Smith”.
- Si encuentra un nombre válido, lo devuelve como autor.

__`enriquecer_autor_wikipedia`__(nombre_autor)

- Busca la página del autor en Wikipedia.
- Intenta extraer el país (ej. “born in France”, “is a British author”).
- Devuelve un diccionario con país, ID de la página y fuente.

__`enriquecer_autor`__(nombre_autor)

- valida si el nombre es correcto.
- Si ya está en caché, lo devuelve directamente.
- Si no, llama a enriquecer_autor_wikipedia y guarda el resultado en caché.

In [5]:
# 5.1 Limpia el título del libro sacándole el sufijo "(Serie #N)" al final, que ensuciaría la búsqueda por título en Wikipedia.
def limpiar_titulo_para_busqueda(titulo:str)->str:
    return re.sub(r'\s*\([^)]*\)\s*$', '', titulo).strip()

#5.2 Valida si un texto parece realmente el nombre de una persona
def es_nombre_valido(nombre:str)->bool: 
    if not nombre:
        return False 

    nombre=nombre.strip()
    if not nombre or nombre=="Autor Desconocido":
        return False

    if len(nombre)>60 or any(c.isdigit() for c in nombre):
        return False

    if any(c in nombre for c in ['@','#','$','%','*','_','~','^', '(', ')', '/']):
        return False

    return bool(re.search(r"^[A-Za-zÀ-ÿ\.\-' ]+$", nombre))

#5.3 Wikipedia busca el título del libro y extrae el autor del resumen
def buscar_autor_por_titulo_wikipedia(titulo_libro: str)-> Optional[str]:
    titulo_busqueda = limpiar_titulo_para_busqueda(titulo_libro)
    parametros = {
        'action': 'query',
        'generator': 'search',
        'gsrsearch': titulo_busqueda,
        'gsrlimit': 5,
        'prop': 'extracts',
        'exintro': 1,
        'explaintext': 1,
        'format': 'json',
        'utf8': 1
    }

    try:
        respuesta=sesion.get('https://en.wikipedia.org/w/api.php', params=parametros, timeout=10)
        respuesta.raise_for_status()
        paginas=respuesta.json().get('query',{}).get('pages',{})
        if not paginas:
            return None

        patrones_autor = [
            r'is\s+an?\s+(?:novel|book|memoir|travelogue)\s+(?:by|written by)\s+([A-Z][a-zA-Z\.\-]+(?:\s+[A-Z][a-zA-Z\.\-]+){1,3})',
            r'written\s+by\s+([A-Z][a-zA-Z\.\-]+(?:\s+[A-Z][a-zA-Z\.\-]+){1,3})',
            r'\bby\s+([A-Z][a-zA-Z\.\-]+(?:\s+[A-Z][a-zA-Z\.\-]+){1,3})[\.,]'
        ]
####aqui lo dejo
        for datos_pagina in paginas.values():
                extracto=datos_pagina.get('extract','')
                if not extracto:
                    continue
                for patron in patrones_autor:
                    coincidencia=re.search(patron, extracto)
                    if coincidencia:
                        nombre_candidato=coincidencia.group(1).strip()
                        if es_nombre_valido(nombre_candidato):
                            return nombre_candidato
        return None

    except requests.exceptions.HTTPError as e:
        print(f"Wikipedia (por título): error HTTP {e.response.status_code} para '{titulo_busqueda}'")
    except requests.exceptions.Timeout:
        print(f"Wikipedia (por título): tiempo de espera agotado para '{titulo_busqueda}'")
    except requests.exceptions.RequestException as e:
        print(f"Wikipedia (por título): error de red para '{titulo_busqueda}': {e}")
    return None

#5.4 Con el nombre validado del autor, busca su página de Wikipedia y extrae el país
def enriquecer_autor_wikipedia(nombre_autor: str) -> Optional[Dict]:
    parametros_busqueda = {
        'action': 'query',
        'generator': 'search',
        'gsrsearch': nombre_autor,
        'gsrlimit': 1,
        'prop': 'extracts',
        'exintro': 1,
        'explaintext': 1,
        'format': 'json',
        'utf8': 1
    }

    try:
        respuesta=sesion.get('https://en.wikipedia.org/w/api.php', params=parametros_busqueda, timeout=10)
        respuesta.raise_for_status()
        paginas=respuesta.json().get('query',{}).get('pages',{})
        if not paginas:
            return None

        datos_pagina=next(iter(paginas.values()))
        titulo_pagina=datos_pagina.get('title',nombre_autor)
        extracto=datos_pagina.get('extract','')
        
        patrones_pais=[
            r'born\s+in\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)?)',
            r'nationality\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)?)',
            r'is\s+an?\s+([A-Z][a-z]+)\s+(?:author|writer|novelist|poet)',
            r'from\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)?)'
        ]
        for patron in patrones_pais:
            coincidencia=re.search(patron, extracto,re.IGNORECASE)
            if coincidencia:
                pais=coincidencia.group(1).split(',')[0].split('.')[0].strip()
                return {'pais':pais, 'api_id':titulo_pagina, 'api_source':'wikipedia'}

        return {'pais':None, 'api_id':titulo_pagina, 'api_source':'wikipedia'}

    except requests.exceptions.HTTPError as e:
        print(f"Wikipedia (autor): error HTTP {e.response.status_code} para '{nombre_autor}'")
    except requests.exceptions.Timeout:
        print(f"Wikipedia (autor): tiempo de espera agotado para '{nombre_autor}'")
    except requests.exceptions.RequestException as e:
        print(f"Wikipedia (autor): error de red para '{nombre_autor}': {e}")
    return None

#5.5 funcion principal de enriquecimiento
def enriquecer_autor(nombre_autor: str) -> Optional[Dict]:
    if not es_nombre_valido(nombre_autor):
        print(f"Autor '{nombre_autor}' no es un nombre válido. Se descarta.")
        return None

    if nombre_autor in cache_autores:
        return cache_autores[nombre_autor]

    resultado = enriquecer_autor_wikipedia(nombre_autor)
    cache_autores[nombre_autor] = resultado
    return resultado

 # 6 Persistencia en BD y guardado de respaldos CSV
guarda y relaciona categorías, autores y libros en la base de datos, evitando duplicados.

__`obtener_id_categoria(conn, nombre_categoria)`__

- Busca si la categoría ya existe en la tabla categorias.
- Si existe, devuelve su ID.
- Si no existe, la inserta y devuelve el nuevo ID.

__`obtener_id_autor(conn, nombre_autor, datos_autor)`__

- Busca si el autor ya existe en la tabla autores.
- Si existe y hay nueva información (país, ID de API, fuente), actualiza los datos.
- Si no existe, inserta con los datos disponibles y devuelve el nuevo ID.

__`guardar_libro(conn, libro, id_categoria, id_autor)`__

- Revisa si el libro ya está guardado (usando la URL como identificador único).
- Si ya existe, solo asegura que la relación con el autor esté registrada.
- Si no existe, inserta en la tabla libros y crea la relación con el autor en la tabla  autor_libro.

In [6]:
def obtener_id_categoria(conn, nombre_categoria:str)->int:
    cursor=conn.cursor()
    cursor.execute("SELECT id_categoria FROM categorias WHERE nombre_categoria = ?",(nombre_categoria,))
    fila=cursor.fetchone()
    if fila:
        return fila[0]

    cursor.execute("INSERT INTO categorias (nombre_categoria) VALUES (?)",(nombre_categoria,))
    conn.commit()
    return cursor.lastrowid

def obtener_id_autor(conn,nombre_autor:str, datos_autor:Optional[Dict])-> int:
    cursor=conn.cursor()
    cursor.execute("SELECT id_autor FROM autores WHERE nombre_autor=?",(nombre_autor,))
    fila=cursor.fetchone()
    if fila:
        id_autor = fila[0]
        # Actualizar país y datos de API si tenemos nueva información
        if datos_autor:
            pais = datos_autor.get('pais')
            api_id = datos_autor.get('api_id')
            api_source = datos_autor.get('api_source')
            if pais or api_id or api_source:
                cursor.execute('''
                    UPDATE autores SET 
                        pais_autor = COALESCE(?, pais_autor),
                        api_external_id = COALESCE(?, api_external_id),
                        api_source = COALESCE(?, api_source)
                    WHERE id_autor = ?
                ''', (pais, api_id, api_source, id_autor))
                conn.commit()
        return id_autor

    pais=None
    api_id=None
    api_source=None

    if datos_autor:
        pais=datos_autor.get('pais')
        api_id=datos_autor.get('api_id')
        api_source=datos_autor.get('api_source')
    
    cursor.execute('''
            INSERT INTO autores (nombre_autor,pais_autor,api_external_id,api_source)
            VALUES(?, ?, ?, ?)
        ''',(nombre_autor,pais,api_id,api_source))
    conn.commit()
    return cursor.lastrowid

#Evita duplicado por URL
def guardar_libro(conn,libro:Dict, id_categoria:int, id_autor:int):
    cursor=conn.cursor()
    cursor.execute("SELECT id_libro FROM libros WHERE url_libro=?",(libro['url'],))
    fila=cursor.fetchone()

    if fila:
        id_libro=fila[0]
        cursor.execute(
            "INSERT OR IGNORE INTO autor_libro(id_libro,id_autor) VALUES (?,?)",
            (id_libro,id_autor)
        )
        conn.commit()
        return

    cursor.execute('''
        INSERT INTO libros (titulo_libro,precio_libro,calificacion_libro,categoria_id,url_libro)
        VALUES (?,?,?,?,?)
    ''',(libro['titulo'],libro['precio'],libro['calificacion'],id_categoria,libro['url']))
    conn.commit()

    id_libro=cursor.lastrowid
    cursor.execute("INSERT INTO autor_libro (id_libro, id_autor) VALUES (?,?)", (id_libro,id_autor))
    conn.commit()

# 7 Proceso principal de scraping con checkpoint en CSV
Para no perder nada si hay un error, guardamos los libros en un servidor.
Al CSV va la informacion cargada, y asi­ la red no vuelve a ser scrapeada.
***

__Archivos definidos__

__`scraping_progreso.json:`__ guarda el estado del scraping (categorías ya procesada y libros guardados).

__`libros_extraidos.csv:`__ archivo final con todos los libros extraídos.

__Funciones de progreso__

__`cargar_progreso():`__ abre el archivo JSON si existe y devuelve el estado; si no, empieza desde cero.

__`guardar_progreso:`__ guarda el estado actual en el JSON.

__`scrapear_todo:`__ (continuar=True)

- Empieza el scraping desde la página principal.
- Si continuar=True, retoma desde donde quedó (no repite categorías ya procesadas).
- Recorre cada categoría, extrae datos y guarda cada libro.
- Busca el autor (en Wikipedia, luego en la API, y si no tiene, marca como “Autor Desconocido”).
- Actualiza el progreso y guarda el caché de autores en cada categoría procesada.
- imprime cuántos libros se guardaron y exporta todo a CSV.

__`exportar_libros_csv():`__
- Consulta la base de datos todos los libros con sus categorías y autores.
- Guarda en libros_extraidos.csv como respaldo.
- Muestra cuántos registros se exportaron.

In [ ]:
ARCHIVO_PROGRESO='datos/scraping_progreso.json'
ARCHIVO_LIBROS_CSV='datos/libros_extraidos.csv'

#7.1 Carga el estado del scraping desde un archivo JSON.
def cargar_progreso():
    if os.path.exists(ARCHIVO_PROGRESO):
        with open(ARCHIVO_PROGRESO,'r')as f:
            return json.load(f)
    return{'categorias_procesadas':[],'libros_guardados':0}

#7.2 guardado del progreso
def guardar_progreso(progreso):
    with open(ARCHIVO_PROGRESO,'w') as f:
        json.dump(progreso,f)

# Ejecuta el scraping completo. Si continuar=True, retoma desde donde quedó.
def scrapear_todo(continuar:bool=True):
    url_principal="https://books.toscrape.com/index.html"
    progreso = cargar_progreso() if continuar else {'categorias_procesadas':[],'libros_guardados':0}

    conexion=obtener_conexion()

# Obtener Categorias
    categorias=extraer_categoria(url_principal)
# Lista de todas las categorias
    lista_categorias=list(categorias.items())

# Si ya hay categorías procesadas, las saltamos
    if continuar:
        procesadas=set(progreso.get('categorias_procesadas',[]))
        lista_categorias = [(nom, url) for nom,url in lista_categorias if nom not in procesadas]

    total_libros_guardados= progreso.get('libros_guardados',0)

    for nombre_categoria,url_categoria in lista_categorias:
        print(f"\n Procesando categoria: {nombre_categoria}")
        id_catategoria = obtener_id_categoria(conexion,nombre_categoria)
        libros = extraer_libros_categoria(url_categoria,nombre_categoria)

        for libro in libros:
            esperar(0.1)
            #extraer el autor desde el nombre del libro desde la pagina del libro
            autor_nombre= buscar_autor_por_titulo_wikipedia(libro['titulo'])
            #si no se encuentra intenta extrarer el titulo
            if autor_nombre: #a la API (si autor_nombre sigue siendo None, no hace falta esperar nada)
                esperar(0.2)

            # 3. Si seguimos sin nada válido, queda como desconocido
            #    (y NO llamamos a enriquecer_autor con basura)
            if es_nombre_valido(autor_nombre):
                datos_autor=enriquecer_autor(autor_nombre)
            else:
                autor_nombre="Autor Desconocido"
                datos_autor=None
            
            id_autor=obtener_id_autor(conexion,autor_nombre,datos_autor)
            guardar_libro(conexion,libro,id_catategoria,id_autor)

            total_libros_guardados+=1
            pais_mostrado=datos_autor.get('pais') if datos_autor else 'N/A'
            print(f"    ✅ {libro['titulo'][:40]}... - Autor: {autor_nombre} (País: {pais_mostrado})")
            
        # Marcar categoria como procesada
        progreso['categorias_procesadas'].append(nombre_categoria)
        progreso['libros_guardados']=total_libros_guardados
        guardar_progreso(progreso)
        guardar_cache_autores()

    conexion.close()
    print(f"\n Scraping completado. Total libros guardados: {total_libros_guardados}")
    
    exportar_libros_csv()

    # Guardar libros extraídos en CSV para respaldo
def exportar_libros_csv():
    conexion=obtener_conexion()
    consulta="""
        SELECT l.id_libro, l.titulo_libro, l.precio_libro, l.calificacion_libro,
                c.nombre_categoria as categoria,
                GROUP_CONCAT(a.nombre_autor, '; ') as autores
        FROM libros l
        JOIN categorias c ON l.categoria_id = c.id_categoria
        JOIN autor_libro al ON l.id_libro = al.id_libro
        JOIN autores a ON al.id_autor = a.id_autor
        GROUP BY l.id_libro
    """
    df=pd.read_sql_query(consulta,conexion)
    conexion.close()
    df.to_csv(ARCHIVO_LIBROS_CSV,index=False, encoding="utf-8")
    print(f"Libros exportados a {ARCHIVO_LIBROS_CSV} ({len(df)} registros).")

: 

 # 8 Ejecutar el scraping
 - Si es la primera vez, ejecuta con continuar=False para empezar desde cero.
 - Si quieres retomar despuÃ©s de una interrupciÃ³n, usa continuar=True (por defecto)

In [8]:
#scrapear_todo(continuar=False)
scrapear_todo(continuar=True)

Contenido HTML obtenido correctamente de https://books.toscrape.com/index.html
Se encontro: 50 Categorias

 Scraping completado. Total libros guardados: 1000
Libros exportados a datos/libros_extraidos.csv (1000 registros).


# 9 Consultas a la Base de Datos
__`Libros con >3 estrellas y <£10:`__ → Busca libros baratos menos de £10 con buena calificación más de 3 estrellas.

__`Autor con peor promedio mínimo 5 libros:`__ → busca autor con peor promedio, pero solo si tiene al menos 5 libros publicados.

__`Categoría con mayor precio promedio:`__
→ Calcula el precio medio de cada categoría y muestra la más cara.

__`Top 5 autores con más libros:`__ →Lista los 5 autores que tienen más libros registrados en la base de datos.

__`País con más libros de rating >3:`__ → Muestra país con más libros con calificación mayor a 3 estrellas.

__`Autores sin país registrado:`__ → Cuenta autores sin información de país guardado.

__`Ranking de autores por país:`__ → en cada país, muestra el autor con más libros y su posición en el ranking.

__`Libros más caros que el promedio de su categoría`__ → muestra libros con mayor precio al promedio de su categoría.

In [12]:
#consulta SQL utiles
def ejecutar_consulta(consulta, parametros=()):
    conexion=obtener_conexion()
    cursor=conexion.cursor()
    cursor.execute(consulta,parametros)
    resultados=cursor.fetchall()
    conexion.close()
    return resultados

# 9.1. Libros con >3 estrellas y <£10
print("1. Libros con >3 estrellas y <£10:")
for fila in ejecutar_consulta("""
    SELECT titulo_libro, precio_libro, calificacion_libro
    FROM libros
    WHERE calificacion_libro > 3 AND precio_libro < 10 
    ORDER BY precio_libro ASC
""")[:10]:
    print(f" -> {fila[0]} - £{fila[1]} ({fila[2]}*)")

# 9.2. Autor con peor promedio (mi­nimo 5 libros)
print ("\n 2. Autor con peor promedio de rating (mi­nimo 5 Libros): ")
resultado = ejecutar_consulta("""
    SELECT a.nombre_autor, AVG(l.calificacion_libro), COUNT(*) as total
    FROM autores a
    JOIN autor_libro al ON a.id_autor=al.id_autor
    JOIN libros l ON al.id_libro=l.id_libro
    GROUP BY a.id_autor 
    HAVING COUNT(*)>=5
    ORDER BY AVG(l.calificacion_libro) ASC
    LIMIT 1 """)

if resultado:
    for fila in resultado:
        print(f"{fila[0]} - Promedio: {fila[1]:.2f}* ({fila[2]} libros)")
else:
    print(" Ningun autor tiene 5 o mas libros en esta categoria")

# 9.3. Categoria con mayor precio Promedio
print("\n 3. Categoria con mayor precio promedio: ")
for fila in ejecutar_consulta("""
    SELECT c.nombre_categoria, AVG(l.precio_libro)
    FROM categorias c
    JOIN libros l ON c.id_categoria=l.categoria_id
    GROUP BY c.id_categoria 
    ORDER BY AVG(l.precio_libro) DESC
    LIMIT 1
    """):
        print(f"    {fila[0]} - £{fila[1]:.2f} promedio")

# 9.4. los 5 autores con mas libros
print("\n 4. Top 5 Autores con mas libros: ")
for i, fila in enumerate(ejecutar_consulta("""
    SELECT a.nombre_autor, COUNT(*) as total
    from autores a
    JOIN autor_libro al ON a.id_autor=al.id_autor
    GROUP BY a.id_autor
    ORDER BY total DESC
    LIMIT 5
"""),1):
        print(f"{i}. {fila[0]} - {fila[1]} libros")

# 9.5. CONSULTA OBLIGATORIA pais que produce mas libros con ranting >3
print("\n5. Pais con mas Libros de rating > 3 estrellas")
resultado= ejecutar_consulta("""
    SELECT a.pais_autor, COUNT(*) as Cantidad
    FROM autores a
    JOIN autor_libro al ON a.id_autor=al.id_autor
    JOIN libros l ON al.id_libro=l.id_libro
    WHERE a.pais_autor IS NOT NULL AND l.calificacion_libro>3
    GROUP BY a.pais_autor 
    ORDER BY cantidad 
    DESC LIMIT 1
""")
if resultado:
    print(f"{resultado[0][0]} - {resultado[0][1]} libros")
else:
    print(" -> No hay datos de pai­s en la BD (API no lo proporciona).")

# 9.6 Autores sin Pais encotrados (registro de los casos no encontrados)
print("\n 6. Autores sin Pais (casos registrados como Null): ")
resultado=ejecutar_consulta(""" 
    SELECT COUNT(*)
    FROM autores
    WHERE pais_autor IS NULL
    """)
print(f" -> {resultado[0][0]} autores sin pais un total de"
    f"-> {ejecutar_consulta ('SELECT COUNT(*) FROM autores')[0][0]} autores.")

# 7. ranking de autores por cantidad de libros, dentro de cada país
print("7. Ranking de autores por país :")
for fila in ejecutar_consulta("""
    SELECT nombre_autor, pais_autor, total_libros, ranking
    FROM (
        SELECT a.nombre_autor, a.pais_autor, COUNT(*) as total_libros,
            RANK() OVER (PARTITION BY a.pais_autor ORDER BY COUNT(*) DESC) as ranking
        FROM autores a
        JOIN autor_libro al ON a.id_autor = al.id_autor
        WHERE a.pais_autor IS NOT NULL
        GROUP BY a.id_autor
    )
    WHERE ranking = 1
    LIMIT 10
"""):
    print(f"   {fila[0]} ({fila[1]}) - {fila[2]} libros - Ranking #{fila[3]} en su país")
    
    print("\n8. Libros más caros que el promedio de su categoría (subconsulta):")
for fila in ejecutar_consulta("""
    SELECT titulo_libro, precio_libro, categoria_id
    FROM libros l
    WHERE precio_libro > (
        SELECT AVG(precio_libro) FROM libros WHERE categoria_id = l.categoria_id
    )
    ORDER BY precio_libro DESC
    LIMIT 10
"""):
    print(f"   {fila[0]} - £{fila[1]:.2f}")

1. Libros con >3 estrellas y <£10:

 2. Autor con peor promedio de rating (mi­nimo 5 Libros): 
DC Comics - Promedio: 2.57* (7 libros)

 3. Categoria con mayor precio promedio: 
    Suspense - £58.33 promedio

 4. Top 5 Autores con mas libros: 
1. Autor Desconocido - 278 libros
2. Natsuki Takaya - 8 libros
3. Marvel Comics - 7 libros
4. DC Comics - 7 libros
5. Cassandra Clare - 4 libros

5. Pais con mas Libros de rating > 3 estrellas
American - 20 libros

 6. Autores sin Pais (casos registrados como Null): 
 -> 388 autores sin pais un total de-> 642 autores.
7. Ranking de autores por país :
   DreamWorks Animation (Amblin Entertainment) - 1 libros - Ranking #1 en su país

8. Libros más caros que el promedio de su categoría (subconsulta):
   Cassandra Clare (American) - 4 libros - Ranking #1 en su país

8. Libros más caros que el promedio de su categoría (subconsulta):
   David Foster Wallace (Amherst College) - 1 libros - Ranking #1 en su país

8. Libros más caros que el promedio de su 

# 10 Prueba de indexacion (antes/despues)
__comparar cuánto tarda una consulta a la base de datos antes y después de crear un índice.__
***
<br> __`Función medir_tiempo:`__ (consulta, descripcion):
- Ejecuta una consulta SQL.
- Mide cuánto tarda en completarse usando time.time().
- Imprime el tiempo junto con una descripción.
***
<br>__`Consulta lenta:`__
- Busca los libros cuyo precio esté entre 20 y 30.
- Al inicio, se ejecuta sin índice, recorre toda la tabla (más lento).
***
<br>__`Plan de ejecución antes y después:`__
- EXPLAIN QUERY PLAN muestra cómo la base va ejecutar la consulta.
- Primero: hace un scan completo de la tabla.
- Luego: crea un índice en la columna precio_libro. para acceder a los registros que tienen la condición
***
<br>__`Medición después del índice:`__
- Se vuelve a medir el tiempo de la misma consulta.
- Ahora debería ser más rápido porque el índice evita revisar fila por fila.


In [ ]:
# 10. Medicion de rendimiento con/sin i­ndices
import time

def medir_tiempo(consulta, descripcion):
    inicio=time.time()
    conexion=obtener_conexion()
    cursor=conexion.cursor()
    cursor.execute(consulta)
    cursor.fetchall()
    conexion.close()
    fin=time.time()
    print (f"{descripcion}: {fin - inicio:.4f} segundos")

# 

consulta_lenta="SELECT * FROM libros WHERE precio_libro BETWEEN 20 AND 30"

print(f"\n Antes de crear el Indice: {consulta_lenta}" )
medir_tiempo(consulta_lenta,"Tiempo Sin Indice")

conexion=obtener_conexion()
plan_antes=conexion.execute(f"EXPLAIN QUERY PLAN {consulta_lenta}").fetchall()
conexion.execute("CREATE INDEX IF NOT EXISTS idx_libros_precio ON libros(precio_libro)")

conexion.commit()

plan_despues = conexion.execute(f"EXPLAIN QUERY PLAN {consulta_lenta}").fetchall()
conexion.close()

print(f"\n Medicion Despues del i­ndice: {consulta_lenta}" )
medir_tiempo(consulta_lenta," Tiempo con Indice") 

print(f"\n Explicacion: El i­ndice reduce el tiempo de busqueda al permitir acceso directo a las filas que cumplen el rango de precios.")


 Antes de crear el Indice: 
Tiempo Sin Indice: 0.0191 segundos

 Medicion Despues del i­ndice: 
 Tiempo con Indice: 0.0011 segundos

 Explicacion: El i­ndice reduce el tiempo de busqueda al permitir acceso directo a las filas que cumplen el rango de años.
